# Stage 04 / Lesson 01: Semantic Chunking Fundamentals
**Course Lecture:** Lecture 37: Semantic Chunking With RAG  
**Section:** Section 8: Advanced Chunking And Preprocessing Techniques

---

## 🧭 Architectural Overview: The Failure of Fixed-Size Chunking & The Semantic Solution

In Stages 01 through 03, document pre-processing relied strictly on deterministic token/character splitting algorithms such as **`RecursiveCharacterTextSplitter`**.

While computationally trivial ($O(N)$ execution), fixed-size chunking introduces severe architectural flaws into production RAG pipelines:

### 1. The Core Pathologies of Fixed-Size Splitting:
- **Semantic Mutilation:** If an atomic proposition begins at character 420 and concludes at character 580 within a 500-character window, the thought is severed across two independent vectors. Neither chunk carries sufficient context to satisfy downstream retrieval.
- **Vector Smearing (Contextual Contamination):** When arbitrary character counts bundle unrelated subject matter (e.g., software engineering documentation immediately followed by geographic trivia), the resulting dense embedding represents an uninterpretable weighted average in vector space.
- **Inflexible Granularity:** Simple ideas are artificially padded, while dense multidimensional paragraphs are truncated.

### 2. The Semantic Chunking Philosophy:
Semantic chunking dynamically partitions text based on **semantic continuity** rather than arbitrary physical length.
Chunks produced under this paradigm exhibit three vital engineering properties:
1. **Self-Contained:** Every chunk encapsulates a complete, resolvable proposition.
2. **Contextually Rich:** Internal semantic coherence maximizes vector signal-to-noise ratio.
3. **Logically Separated:** Boundaries occur exclusively at semantic inflection points (breakpoints).

## 1. Environment & Setup
Initialize the embedding model and verify dependencies.

In [1]:
import re
import numpy as np
import matplotlib.pyplot as plt
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# initialize local embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("✅ Environment initialized successfully.")
print(f"📦 Embedding model: sentence-transformers/all-MiniLM-L6-v2")

✅ Environment initialized successfully.
📦 Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 2. Experimental Corpus & Pathological Baseline Demonstration
Define the experimental corpus introduced in Lecture 37, containing two distinct conceptual clusters:
- **Cluster A (Software Frameworks):** Sentences regarding LangChain, RAG architecture, and Generative AI workflows.
- **Cluster B (Geography & Architecture):** Sentences regarding Paris, France, and the Eiffel Tower.

In [2]:
# experimental multi-topic document
corpus_text = (
    "LangChain is a framework to build RAG applications. "
    "LangChain can also be used to build Generative AI applications and multi-agent workflows. "
    "Paris is the capital of France. "
    "The Eiffel Tower is located in Paris, France and is a popular destination."
)

print("--- RAW CORPUS TEXT ---")
print(corpus_text)

# evaluate baseline behavior with RecursiveCharacterTextSplitter
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=110, chunk_overlap=15)
fixed_chunks = fixed_splitter.split_text(corpus_text)

print(f"\n⚠️ Fixed-Size Splitter generated {len(fixed_chunks)} chunks (chunk_size=110):\n")
for idx, chunk in enumerate(fixed_chunks, start=1):
    print(f"  Chunk [{idx}] ({len(chunk)} chars): '{chunk}'")

--- RAW CORPUS TEXT ---
LangChain is a framework to build RAG applications. LangChain can also be used to build Generative AI applications and multi-agent workflows. Paris is the capital of France. The Eiffel Tower is located in Paris, France and is a popular destination.

⚠️ Fixed-Size Splitter generated 3 chunks (chunk_size=110):

  Chunk [1] (101 chars): 'LangChain is a framework to build RAG applications. LangChain can also be used to build Generative AI'
  Chunk [2] (105 chars): 'Generative AI applications and multi-agent workflows. Paris is the capital of France. The Eiffel Tower is'
  Chunk [3] (63 chars): 'Tower is located in Paris, France and is a popular destination.'


## 3. The 4-Step Semantic Chunking Algorithm

The mathematical pipeline executes in four deterministic phases:

```mermaid
flowchart TD
    RawDoc["Raw Document Text"] --> S1["Phase 1: Document Segmentation
(Split into discrete propositions/sentences)"]
    S1 --> S2["Phase 2: Sentence Embeddings
(Compute dense vectors E_1, E_2, ..., E_n)"]
    S2 --> S3["Phase 3: Pairwise Adjacent Cosine Similarity
Sim(E_i, E_i+1) = (E_i · E_i+1) / (||E_i|| ||E_i+1||)"]
    S3 --> S4["Phase 4: Threshold Boundary Detection
If Sim < Threshold -> Breakpoint (New Chunk)
If Sim >= Threshold -> Merge into Active Chunk"]
    S4 --> FinalChunks["Pure, Self-Contained Semantic Chunks"]
```

### Phase 1: Document Segmentation
Partition raw document stream into individual sentence units.

In [3]:
# segment corpus into sentence units
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", corpus_text) if s.strip()]

print(f"✅ Phase 1: Segmented into {len(sentences)} sentence units:")
for idx, s in enumerate(sentences, start=1):
    print(f"  S{idx}: '{s}'")

✅ Phase 1: Segmented into 4 sentence units:
  S1: 'LangChain is a framework to build RAG applications.'
  S2: 'LangChain can also be used to build Generative AI applications and multi-agent workflows.'
  S3: 'Paris is the capital of France.'
  S4: 'The Eiffel Tower is located in Paris, France and is a popular destination.'


### Phase 2: Dense Vector Representation
Compute dense vector representations for each sentence unit.

In [4]:
# compute sentence embeddings
sentence_embeddings = [np.array(embeddings.embed_query(s)) for s in sentences]

print(f"✅ Phase 2: Generated {len(sentence_embeddings)} dense sentence vectors.")
print(f"Vector Dimensions: {sentence_embeddings[0].shape[0]}")

✅ Phase 2: Generated 4 dense sentence vectors.
Vector Dimensions: 384


### Phase 3: Consecutive Pairwise Cosine Similarity
Calculate cosine similarity across adjacent sentence pairs:
$$\text{Sim}(S_i, S_{i+1}) = \frac{\mathbf{u}_i \cdot \mathbf{u}_{i+1}}{\|\mathbf{u}_i\|_2 \|\mathbf{u}_{i+1}\|_2}$$

In [5]:
def cosine_similarity(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

similarities = []
for i in range(len(sentence_embeddings) - 1):
    sim = cosine_similarity(sentence_embeddings[i], sentence_embeddings[i + 1])
    similarities.append(sim)
    print(f"Sim(S{i+1}, S{i+2}): {sim:.4f}")

# observe the significant semantic drop between S2 (LangChain) and S3 (Paris)
semantic_drop = similarities[0] - similarities[1]
print(f"\n📉 Semantic Drop Magnitude between S1-S2 and S2-S3: {semantic_drop:.4f}")

Sim(S1, S2): 0.5493
Sim(S2, S3): -0.0077
Sim(S3, S4): 0.5496

📉 Semantic Drop Magnitude between S1-S2 and S2-S3: 0.5570


### Phase 4: Threshold Boundary Detection & Chunk Formation
Apply threshold filtering to establish chunk boundaries.

In [6]:
threshold = 0.50
semantic_chunks = []
active_chunk = [sentences[0]]

for i, sim in enumerate(similarities):
    if sim >= threshold:
        active_chunk.append(sentences[i + 1])
    else:
        semantic_chunks.append(" ".join(active_chunk))
        active_chunk = [sentences[i + 1]]

if active_chunk:
    semantic_chunks.append(" ".join(active_chunk))

print(f"🎯 Phase 4: Semantic Chunking produced {len(semantic_chunks)} coherent chunks:\n")
for idx, chunk in enumerate(semantic_chunks, start=1):
    print(f"=== Semantic Chunk [{idx}] ===")
    print(f"'{chunk}'\n")

🎯 Phase 4: Semantic Chunking produced 2 coherent chunks:

=== Semantic Chunk [1] ===
'LangChain is a framework to build RAG applications. LangChain can also be used to build Generative AI applications and multi-agent workflows.'

=== Semantic Chunk [2] ===
'Paris is the capital of France. The Eiffel Tower is located in Paris, France and is a popular destination.'


## 4. Visualizing the Semantic Valley (Breakpoint Analysis)
Graph consecutive similarity scores alongside the decision boundary threshold.

In [7]:
pairs = [f"S{i+1}-S{i+2}" for i in range(len(similarities))]

plt.figure(figsize=(7, 3.5))
plt.plot(pairs, similarities, marker='s', color='#1E40AF', linewidth=2.5, markersize=8, label='Pairwise Cosine Similarity')
plt.axhline(y=threshold, color='#DC2626', linestyle='--', linewidth=2, label=f'Decision Threshold ({threshold})')

# annotate semantic drop point
plt.annotate(
    'Semantic Breakpoint\n(Topic Transition)',
    xy=(1, similarities[1]),
    xytext=(1.05, similarities[1] + 0.15),
    arrowprops=dict(facecolor='#DC2626', shrink=0.08, width=1.5, headwidth=6),
    fontweight='bold',
    color='#DC2626'
)

plt.title("Consecutive Sentence Cosine Similarity Profile", fontsize=12, fontweight='bold')
plt.ylabel("Cosine Similarity", fontsize=10)
plt.xlabel("Sentence Transition Pairs", fontsize=10)
plt.ylim(0.0, 1.0)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 5. Architectural Comparison Matrix: Fixed-Size vs. Semantic Chunking
| Architectural Dimension | Fixed-Size Character Splitting | Semantic Chunking |
| :--- | :--- | :--- |
| **Splitting Heuristic** | Character / Token length counting | Embedding distance & semantic similarity |
| **Computational Overhead** | Ultra-low $O(N)$ string slicing | Higher (Requires embedding generation per sentence) |
| **Contextual Integrity** | Low (Frequent thought truncation) | Optimal (Chunks are conceptually self-contained) |
| **Vector Purity (Embedding Quality)** | Degraded by topic mixing (Vector Smearing) | High (Each vector represents a single cohesive concept) |
| **Downstream Impact on RAG** | Higher hallucination risk, noisy retrieval | Superior retrieval precision, cleaner prompt context |
| **Best Applied To** | Raw initial indexing, simple homogeneous text | Complex manuals, multi-topic articles, enterprise RAG |